# 见微知著 - Decoding EEG Movement Imagination

Codebook by Antonia Reul | Contact: areul@uni-osnabrueck.de

**Main research question:** 

    Can a hybrid CNN-Transformer architecture reliably predict imagined, but not executed movements based on EEG recordings?


This Jupyter notebook combines the five main .py files, was developed to test the pipeline all in one place and display plots directly, so that interested readers can understand and work with the code easily.

In [ ]:
import math
import matplotlib.pyplot as plt
import mne 
import numpy as np
from mne.datasets import eegbci
from mne.io import concatenate_raws, read_raw_edf
from mne.preprocessing import ICA
from sklearn.metrics import cohen_kappa_score, confusion_matrix
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LambdaLR
from torch.utils.data import Dataset, DataLoader
from typing import tuple

seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

/opt/miniconda3/envs/dl26/lib/python3.12/site-packages/mne/externals/tempita/__init__.py:35: DeprecationWarning: 'cgi' is deprecated and slated for removal in Python 3.13
  import cgi


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)

## Download_data.py
Helper .py file to ensure the code works on the HPC.

In [ ]:
def download():
    BAD = {88, 89, 92, 100}
    runs = [4, 8, 12]
    print("Starting mass download... this may take a while.")

    for s in range(1, 110):
        if s in BAD:
            continue
        try:
            eegbci.load_data(subjects=s, runs=runs, update_path=True)
            print(f"Downloaded subject {s}")
        except Exception as e:
            print(f"Failed to download subject {s}: {e}")

    print("All available data downloaded to ~/mne_data")

download()

## Preprocess.py
_Importing and preprocessing the data of each subject_

The Physionet dataset contains EEG data recorded for different task categories. Since the focus of this project is on movement imagery, the runs for left vs. right fist imagination (runs=[4, 8, 12]) have been selected. Using the runs for both fist vs. both feet imagination could have also been possible or comparing movement imagination vs. movement execution data, although that would alter the research question of this project. Data has been selected from 105 subjects. The complete dataset contains 109 subjects, but four subjects have repeatedly been reported to be unsuitable for working with (more below). Each subject performed about 15 task trials per run, so approximately 45 trials in total (3 runs). This leads to data with about 4700 trials of shape (64, 481) and with a 80/20 test-train split, about 3800 training trials. 

#### Data Exclusion
Subjects 88, 89, 92 and 100 have been excluded since subject 89 had incorrect labels and the sampling rate of the other three subjects was recorded at 128 Hz instead of the original 160 Hz.

In [ ]:
try:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(__file__), '..'))
except NameError:
    PROJECT_ROOT = os.getcwd()

if os.path.basename(PROJECT_ROOT) == 'src':
    PROJECT_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, '..'))

print(f"Project Root identified as: {PROJECT_ROOT}")

LOCAL_DATA_ROOT = "/home/student/a/areul/mne_data/MNE-eegbci-data/files/eegmmidb/1.0.0/"

"""A plotted analysis of the dataset can be found in a separate
Jupyter notebook in the folder 'notebooks'

Due to the MNE library still depending on some NumPy1. functions but newer
scipy commands, numpy=1.26.4 and scipy=1.12.0 are suggested for running 
this code successfully in an environment."""

# Dataset documentation: https://www.physionet.org/content/eegmmidb/1.0.0/
# MNE: https://mne.tools/stable/generated/mne.datasets.eegbci.load_data.html


def load_subject_data(subject_id: int, runs=[4, 8, 12], preload=True, baseline=None, data_root: str = LOCAL_DATA_ROOT) -> Tuple[np.ndarray, np.ndarray]:
    # Loading the raw data file for a single subject
    print('Checkpoint 1: Loading EDF files for subject {subject_id}')
    paths = []
    for run in runs:
        subject_folder_name = f"S{subject_id:03d}"
        filename_inside_folder = f"S{subject_id:03d}R{run:02d}.edf"
        full_file_path = os.path.join(data_root, subject_folder_name, filename_inside_folder)
        
        if os.path.exists(full_file_path):
            paths.append(full_file_path)
        else:
            print(f"ERROR: Could not find expected file for subject {subject_id}, run {run} at: {full_file_path}")
            return None, None

    if not paths:
        print(f"CRITICAL ERROR: No EDF files were successfully located for subject {subject_id}.")
        return None, None

    raw = concatenate_raws([read_raw_edf(p, preload=True) for p in paths])

    print('Checkpoint 2: Annotations and Montage')
    events, event_id = mne.events_from_annotations(raw, event_id=dict(T0=1, T1=2, T2=3))
    eegbci.standardize(raw)    
    # 10-10 system used excluding Nz, F9/F10, ...
    raw.set_montage(mne.channels.make_standard_montage('standard_1005'))

    # Apply notch and high-pass filter (2nd needed for ICA)
    print('Checkpoint 3: Filtering')
    raw.notch_filter(freqs=[60]) # Nyquist freq 80 Hz (160/2)
    raw_for_ica = raw.copy().filter(l_freq=4.0, h_freq=None)
    # 4 Hz removes drift and blinks

    # Apply ICA to filtered copy 
    print('Checkpoint 4: Starting ICA')
    ica = ICA(n_components=0.99, random_state=42, method='fastica')
    ica.fit(raw_for_ica)

    # Find and apply components to original raw data
    # using frontal-polar electrodes closest to eyes 
    # -> most sensitive to EOG signals
    eog_ind = ica.find_bads_eog(raw, ch_name=['Fp1', 'Fp2'])
    # Debugging case if MNE returns list in list
    if isinstance(eog_ind, list) and len(eog_ind) > 0 and isinstance(eog_ind[0], list):
        eog_ind = eog_ind[0]
    eog_ind = eog_ind[0]

    ica.apply(raw, exclude=set(eog_ind))

    # Define event ID
    print('Checkpoint 5: Epoching...')
    event_id = {"left": 2, "right": 3}

    # Epoching into 4 s windows
    epochs = mne.Epochs(raw, events=events, event_id=event_id, 
                                tmin=0.5, tmax=3.5, baseline=baseline, 
                                preload=preload, reject=dict(eeg=1000e-6), 
                                flat=dict(eeg=1e-7))

    # Extract data and labels
    print('Checkpoint 6: Finalizing data')
    X = epochs.get_data().astype(np.float32) * 1e6 # conversion to µV
    # (n_epochs, n_channels, n_samples)
        
    # Normalize data by z-score normalization 
    # Calculate mean and std across epoch and time dim per channel
    mu = np.mean(X, axis=(0, 2), keepdims=True)
    sd = np.std(X, axis=(0, 2), keepdims=True)
    X = (X - mu) / (sd + 1e-8)

    y = epochs.events[:, 2] # Event IDs from 3rd column (2 for left, 3 for right)

    # Raw event IDs do not start at 0 which PyTorch classification losses expect 
    # -> map to (0, 1) binary scale
    label_map = {2: 0, 3: 1}

    # Target vector y
    # T1 = 0 and T2 = 1
    y = np.array([label_map[label] for label in y])

    return X, y
    # (batch, channels, samples)


def run_preprocessing():
    processed_dir = os.path.join(PROJECT_ROOT, 'data', 'processed')
    x_path = os.path.join(processed_dir, 'eeg_X_processed.npy')
    y_path = os.path.join(processed_dir, 'eeg_y_processed.npy')

    if os.path.exists(x_path) and os.path.exists(y_path):
        print("Processed data already exists. Skipping preprocessing to save time.")
        return
    
    print("Processed data not found. Starting full preprocessing pipeline.")
    os.makedirs(processed_dir, exist_ok=True)

    BAD = {88, 89, 92, 100}
    subject_ids = [s for s in range(1, 110) if s not in BAD] 

    X_all, y_all = [], []
    for s in subject_ids:
        print(f"Processing subject {s}...")
        try:
            X, y = load_subject_data(s)
            X_all.append(X)
            y_all.append(y)
        except Exception as e:
            print(f"Skipping subject: {s}: {e}")

    if X_all and y_all:
        X = np.concatenate(X_all, axis=0)
        y = np.concatenate(y_all, axis=0)

        np.save(os.path.join(processed_dir, 'eeg_X_processed.npy'), X)
        np.save(os.path.join(processed_dir, 'eeg_y_processed.npy'), y)

        print(f"Finished! 'eeg_X_processed.npy' and 'eeg_y_processed.npy' saved to {processed_dir}")

    else:
        print("CRITICAL FAILURE: No subjects were successfully processed. Please verify the file paths and naming conventions against the actual data structure.")


In [ ]:
run_preprocessing()

## Dataset.py

In [ ]:
class PreprocessedDataset(Dataset):
    """
    Initialize the dataset loader for the eegbci dataset
    
    Parameters:
    - subject_ids: List of subject IDs to load
    - runs: List of run numbers (e. g. [4] for left vs. right hand)
    - preload: Whether to load data into memory
    - baseline: Baseline correction
    """
    
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        # Return total number of samples
        return len(self.X)

    def __getitem__(self, idx):
        # Access a single sample by index
        return torch.tensor(self.X[idx], dtype=torch.float32), torch.tensor(self.y[idx], dtype = torch.float32)
    

## Datamodule.py
_Creating data loaders to organize the data into small batches_

In order to check for exploding or vanishing gradients and other unwanted phenomena during the training, the dataset is split into a train, validation and test set with a 70%/10%/20% distribution.

In [ ]:
def create_dataloaders(batch_size):
    """
    Create data loaders to organize the data into small batches
    """

    processed_dir = os.path.join(PROJECT_ROOT, 'data', 'processed')

    try:
        X = np.load(os.path.join(processed_dir, 'eeg_X_processed.npy'))
        y = np.load(os.path.join(processed_dir, 'eeg_y_processed.npy'))
        
    except FileNotFoundError:
        raise FileNotFoundError(f"Processed files not found in {processed_dir}. Please run preprocess.py first!")

    full_dataset = PreprocessedDataset(X, y)
    indices = np.arange(len(X))

    # 80 % of subjects used for training, 20 % for testing
    train_val_subjects, test_subjects = train_test_split(indices, test_size=0.2, random_state=42)

    # 70 % of subjects used for training, 10 % for validation during training
    train_subjects, val_subjects = train_test_split(train_val_subjects, test_size=0.125, random_state=42)

    print(f"Subjects, train: {len(train_subjects)}, val: {len(val_subjects)}, test: {len(test_subjects)}")

    # Some other projects use batch size of 16 with the PhysioNet dataset
    train_loader = DataLoader(Subset(full_dataset, train_subjects), batch_size=batch_size, shuffle=True) 
    val_loader = DataLoader(Subset(full_dataset, val_subjects), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(Subset(full_dataset, test_subjects), batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader

## Model.py
_Different classes for both the main and comparison models_

This was the first draft as a baseline, however, it quickly became clear that MLPs are unsuitable for working with EEG data. The hidden layers require lots of parameters (EEG data contains many features), so the MLP is very likely to just overfit (memorize noise of) the EEG data, which has a low signal-to-noise ration, rather than actually learn meaningful representations. To work with the EEG data, it would have to get flattened, which effectively destroys spatial and temporal information by just collapsing them onto one dimension. Instead, global average pooling (GAP) can also be used. To reduce the dimensionality of the vector, GAP takes the average of the time dimension for each channel. In order to still have a reasonable baseline, a one layer CNN is used as the main baseline model. For full transparent documentation the poor MLP was included in this notebook, but deliberately outcommented in the model.py code to save computational power - with a couple of million parameters approximately, it can be quite computationally expensive to train (also it has not been checked in the pipeline_test and might contain bugs). Since this is a university project and I have worked on it, I decided to leave it in here, nevertheless, but to not effectively use it in the analysis.

In [ ]:
class PoorMLP(nn.Module):
    """
    A small and simple baseline model to compare the main model to
    
    Parameters:
    - in_dim: Dimension of the input data
    - num_classes: Number of classes (2)
    - hidden_dim: Hidden dimension size of MLP
    - dropout: Dropout rate, here large since comp. expensive
    """

    def __init__(self, in_dim, num_classes=2, hidden_dim=128, dropout=0.5) -> None:
        # EEG datasets small & noisy -> strong dropout suggested
        # 64 channels/electrodes -> 1:1 mapping
        super(PoorMLP, self).__init__()
        
        self.layer_1 = nn.Linear(in_dim, hidden_dim)
        self.layer_2 = nn.Linear(hidden_dim, hidden_dim // 2)
        # Enforce dense representations (hidden_dim // 2)
        self.layer_out = nn.Linear(hidden_dim // 2, num_classes)
        self.dropout = nn.Dropout(dropout)
        self.elu = nn.ELU()
        
    def forward(self, x):
        x = torch.mean(x, dim=2) # (batch, channels)
        x = self.elu(self.layer_1(x))
        x = self.dropout(x)
        x = self.elu(self.layer_2(x))
        x = self.dropout(x)
        x = self.layer_out(x)
        return x

Instead of using a MLP for the classification task, a simple CNN can be used. Using convolutional and pooling layers, CNNs act as efficient feature extractors. However due to their local inductive bias, they struggle with capturing global relationships, which is why research has shifted to focusing on hybrid architectures in recent years, which will be described further below.

In [ ]:
class BaselineCNN(nn.Module):
    """
    Baseline model for main model performance comparison

    Parameters:
    - num_classes: Number of classes
    """
    
    def __init__(self, num_classes=1):
        super(BaselineCNN, self).__init__()

        self.conv1 = nn.Conv1d(in_channels=64, out_channels=16, kernel_size=25)
        self.pool = nn.AdaptiveAvgPool1d(1) # Reduce time dimension to 1
        self.fc = nn.Linear(16, num_classes)
        self.elu = nn.ELU()

    def forward(self, x):
        x = self.elu(self.conv1(x))     # (batch, 16, samples-24)
        x = self.pool(x).squeeze(-1)    # (batch, 16)
        x = self.fc(x)                  # (batch, 2)
        return x.squeeze(-1)

When analysing EEG data, temporal (frequency) should occur before spatial filtering, i. e. detecting which electrodes are important. If 1D convolutional layers are used, instead of 2D, the information from all electrodes get mixed up. Instead, the goal should be to analyse the time dimension for each electrode independently, using a temporal filter first, as has been done in popular networks like EEGNet (Lawhern et al., 2018). Further, ELU is used instead of ReLU since it allows for negative values which is useful when dealing with zero-mean oscillatory signals. After every convolution, batch normalization is used to ensure stability since EEG signals are noisy and highly variable between subjects. Only 16 temporal filters (F1) are used since they have been shown to generally be sufficient to cover primary frequency ranges of interest in MI, as has been shown in the EEGNet and ShallowConvNet papers. Using too many filters, like 64, may lead to the model drastically overfitting. Besides that, for every temporal filter D spatial filters are created, so that the model can look for different spatial topographies for the same frequency. Moreover, the pool size for the global average pooling has been chosen since it is the standard balance used in the EEGNet and ShallowConvNet literature for sampling rates between 128 Hz and 256 Hz. One could also experiment with the values 8 and 2 instead, to compare how model performance improves or worsens.

In [ ]:
class CNN(nn.Module):
    """
    CNN to transform the continuous time-series values into discrete
    token sequences/localized patches and to capture local relationships 
    before input is passed to the Transformer

    Parameters:
    - n_channels: Number of channels, 64 in our dataset
    - num_classes: Number of classes, 2
    - emb_dim: Dimension of embeddings
    - fs: Sampling rate
    """

    def __init__(self, n_channels=64, num_classes=2, emb_dim=128, fs=160):
        super(CNN, self).__init__()

        F1 = 40 # Number of temporal filters
        D = 2 # Depth multiplier for spatial filters
        k_t = fs // 10 # Temporal kernel size, here: 16

        # Temporal Convolution: Learn frequency/band-pass filters
        # input: (batch, 1, n_channels, time)
        self.temp_conv = nn.Conv2d(1, F1, (1, k_t), padding=(0, k_t // 2), bias=False)
        self.bn1 = nn.BatchNorm2d(F1)

        # Spatial Convolution: Learn spatial filters
        self.spat_conv = nn.Conv2d(F1, F1 * D, (n_channels, 1), groups=F1, bias=False)
        self.bn2 = nn.BatchNorm2d(F1 * D)

        self.elu = nn.ELU(True)
        self.pool = nn.AvgPool2d(kernel_size=(1, 4)) # Pool along time dimension
        self.embedding_layer = nn.Linear(F1 * D, emb_dim)

    def forward(self, x):
        # x input shape: (batch, n_channels, time)
        # Reshape to (batch, 1, n_channels, time) for Conv2d
        x = x.unsqueeze(1)

        x = self.temp_conv(x)
        x = self.bn1(x)
        x = self.elu(x)

        x = self.spat_conv(x)
        x = self.bn2(x)
        x = self.elu(x)

        x = self.pool(x) # (batch, F1*D, 1, time_reduced)

        # Transform to 3D for transformer
        x = x.squeeze(2) # (batch, F1*D, time_reduced)
        x = x.transpose(1, 2) # (batch, time_reduced, F1*D)

        x = self.embedding_layer(x) # (batch, time_reduced, emb_dim)

        return x

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Positional encoding so important temporal information 
    is not lost when input is passed onto Transformer

    Parameters:
    - emb_dim: Dimension of embeddings
    - max_patches: Maximum number of patches
    """
    
    def __init__(self, emb_dim, max_patches):
        super(PositionalEncoding, self).__init__()
        self.emb_dim = emb_dim
        assert emb_dim % 2 == 0 # must be even for sin/cos pairs

        # Positional encoding matrix
        pe = torch.zeros(max_patches, emb_dim)

        # Sinusoidal positional encoding
        # Position indices
        pos = torch.arange(0, max_patches, dtype=torch.float).unsqueeze(1) # (max_patches, 1)

        # Division term (tensor of even indices since pose alternate between sin/cos)
        div_term = torch.exp(torch.arange(0, emb_dim, 2).float() * (-math.log(10000.0) / emb_dim))

        pe[:, 0::2] = torch.sin(pos * div_term) # Every second column starting from index 0
        pe[:, 1::2] = torch.cos(pos * div_term) # For all odd indices cosinusoidal values

        # New batch dimension: (1, max_patches, emb_dim)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :] # (batch, seq_len, d_model)
        return x

In [ ]:
class ResidualConnection(nn.Module):
    """
    Residual connection for faster and more efficient Transformer training
    
    Parameters:
    - block: Network the residual connection is applied to (e. g. feed forward block)
    - emb_dim: Embedding dimension
    - dropout: Dropout rate
    """

    def __init__(self, block, emb_dim, dropout=0.1):
        super(ResidualConnection, self).__init__()
        self.block = block
        self.norm = nn.LayerNorm(emb_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.dropout(self.block(self.norm(x)))
        return x


class FeedForwardBlock(nn.Module):
    """
    Feed-forward block of the Transformer, a simple MLP
    
    Parameters:
    - in_dim: Dimension of the input
    - exp_fct: Expansion factor, how much to expand hidden layer
    - dropout: Dropout rate
    """

    def __init__(self, in_dim, exp_fct=4, dropout=0.1):
        super(FeedForwardBlock, self).__init__()

        hidden_dim = in_dim * exp_fct

        self.linear_in = nn.Linear(in_dim, hidden_dim)
        self.silu = nn.SiLU()
        self.dropout = nn.Dropout(dropout)
        self.linear_out = nn.Linear(hidden_dim, in_dim)

    def forward(self, x):
        x = self.linear_in(x)
        x = self.silu(x) 
        x = self.dropout(x)
        x = self.linear_out(x)
        return x

Multi-head attention returns a tuple of output and weights, but the residual connection class expects a single tensor. Therefore, a wrapper (class AttentionWrapper) is constructed to ensure that the residual connections receive input of correct dimension. The EncoderBlock class then combines all Transformer elements into the Encoder-only architecture.

In [ ]:
class AttentionWrapper(nn.Module):
    """
    Helper to wrap MultiheadAttention because MHA returns tuples
    but ResidualConnection expects a single tensor.
    """
    def __init__(self, emb_dim, n_heads, dropout):
        super(AttentionWrapper, self).__init__()
        self.mha = nn.MultiheadAttention(emb_dim, n_heads, dropout, batch_first=True)

    def forward(self, x):
        # Only return the output tensor, not the attention weights
        out, _ = self.mha(x, x, x)
        return out


class EncoderBlock(nn.Module):
    """
    Encoder-Only Transformer to capture global relationships using multihead attention
    
    Parameters:
    - emb_dim: Embedding dimension
    - n_heads: Number of heads for MHA
    - dropout: Dropout rate, 0.1 similar to Vaswani et al.
    - expansion: Expansion rate
    """

    def __init__(self, emb_dim, n_heads, dropout=0.1, expansion=4):
        super(EncoderBlock, self).__init__()

        self.attention = AttentionWrapper(emb_dim, n_heads, dropout)
        self.ffn = FeedForwardBlock(emb_dim, expansion, dropout)
        self.residual1 = ResidualConnection(self.attention, emb_dim, dropout)
        self.residual2 = ResidualConnection(self.ffn, emb_dim, dropout)

    def forward(self, x, mask=None):
        x = self.residual1(x)
        x = self.residual2(x)
        return x

In MI, the brain usually produces sustained drops or increases in power, like Mu-rhythm desynchronization, over a few seconds. To ignore short-duration noise evoking single spikes, average pooling is used to observe how active the frequency band was throughout the trial. Sometimes concatenation (pooling_type='concat') is also used in motor imagery classification tasks. Concatenation gives the MLP the decision between learning to ignore the maximum signal if it is too noixy and rely on the averaged part instead, or vice versa.

In [ ]:
class MLPClassifier(nn.Module):
    """
    Final MLP Classifier Layer
    
    Parameters:
    - tr_out_dim: Dimension of transformer output (emb_dim)
    - dropout: Dropout rate
    - pooling_type: Pooling type, here 'avg'
    """

    def __init__(self, tr_out_dim, dropout=0.5, pooling_type='avg'):
        super(MLPClassifier, self).__init__()
        self.pooling_type = pooling_type
        input_dim = tr_out_dim

        self.linear_in = nn.Linear(input_dim, tr_out_dim // 2)
        self.linear_out = nn.Linear(tr_out_dim // 2, 1)
        self.elu = nn.ELU(True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = torch.mean(x, dim=1) # (batch, emb_dim)
        x = self.linear_in(x)
        x = self.elu(x)
        x = self.dropout(x)
        x = self.linear_out(x)

        return x

The model finally returns a logit for every sample in the batch. Large positive values signal a high confidence in class 1, while large negative values signal a high confidence for class 0.

In [ ]:
class EEGClassifier(nn.Module):
    """
    Main CNN-Transformer Model

    Parameters:
    - n_channels: Nmber of EEG input channels (64)
    - emb_dim: Dimension of embeddings
    - max_patches: Max length of time sequences after CNN pooling
    - n_heads: Number of heads for multihead attention
    - dropout: Dropout rate
    - fs: Sampling rate of EEG data, for PhysioNet dataset 160 Hz
    """

    def __init__(self, n_channels=64, emb_dim=128, max_patches=500, n_heads=4, dropout=0.1, fs=160):
        super(EEGClassifier, self).__init__()
    
        self.cnn = CNN(n_channels=n_channels, emb_dim=emb_dim, fs=fs)
        self.positional_encoding = PositionalEncoding(emb_dim, max_patches) 
        self.transformer = EncoderBlock(emb_dim, n_heads, dropout)
        self.final_norm = nn.LayerNorm(emb_dim)
        self.mlp = MLPClassifier(tr_out_dim=emb_dim) 

    def forward(self, x):
        """
        Args: 
            x: Input EEG data (batch, n_channels, n_time_points)
        Returns: 
            logits for binary classification (batch size,)
        """
        x = self.cnn(x) # (batch, n_channels, n_samples) -> (batch, time_reduced, emb_dim)
        x = self.positional_encoding(x) # (batch, time_reduced, emb_dim)
        x = self.transformer(x) # (batch, time_reduced, emb_dim)
        x = self.final_norm(x)
        x = self.mlp(x) # (batch, time_reduced, emb_dim) -> (batch, 1)
        
        return x.squeeze(-1) # (batch, 1) -> (batch,)


## Evaluate.py

In [ ]:
def evaluate(model, val_loader, criterion, device):
    model.eval() # disable dropout and batchnorm
    val_loss, val_correct, val_total = 0.0, 0, 0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device).float().view(-1)
    
            pred = model(batch_X)
            loss = criterion(pred, batch_y)

            val_loss += loss.item()
            predicted_classes = (torch.sigmoid(pred) > 0.5).int()
            val_correct += (predicted_classes == batch_y).sum().item()
            val_total += batch_y.size(0)

            all_preds.extend(predicted_classes.cpu().numpy())
            all_targets.extend(batch_y.cpu().numpy())

    epoch_kappa = cohen_kappa_score(all_targets, all_preds)

    return {
        'loss': val_loss / len(val_loader),
        'acc': 100 * val_correct / val_total,
        'kappa': epoch_kappa
    }


def get_predictions(model, test_loader, device):
    """Collects all true labels and predictions"""
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for batch_X, batch_y in test_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device).float().view(-1)
            pred = model(batch_X)
        
            predicted_classes = (torch.sigmoid(pred) > 0.5).int()
            all_preds.extend(predicted_classes.cpu().numpy())
            all_targets.extend(batch_y.cpu().numpy())

    return np.array(all_targets), np.array(all_preds)


def visualize_predictions(y_true, y_pred, model_name="model", learning_rate=None, save=False):
    """Visualize prediction accuracy with confusion matrices"""

    unique, counts = np.unique(y_pred, return_counts=True)
    print("Prediction Distribution:")
    print(dict(zip(unique, counts)))

    kappa = cohen_kappa_score(y_true, y_pred)
    print(f"Cohen's Kappa: {kappa:.4f}")

    cm = confusion_matrix(y_true, y_pred)

    fig1 = plt.figure(1, figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Left', 'Right'], yticklabels=['Left', 'Right'])
    plt.xlabel('Predicted')
    plt.ylabel('True')
    lr_str = f"lr_{learning_rate:.1e}" 
    plt.title(f'Confusion Matrix: {model_name}, lr = {learning_rate}')

    if save:
        plot_dir = os.path.join(PROJECT_ROOT, 'results', 'plots')
        os.makedirs(plot_dir, exist_ok=True)

        if learning_rate is not None:
            filename = f"{model_name}_dropout.4_{lr_str}_confusion_matrix.png"
        else:
            filename = f"{model_name}_dropout.4confusion_matrix.png"

        save_path = os.path.join(plot_dir, filename)
        plt.savefig(save_path)
        print(f"Confusion matrix saved to: {save_path}")
        plt.close()

    else:
        plt.show()

## Train.py
_Loading and preprocessing the data, passing it through the baseline MLP and then main model, optimizing via the Adam optimizer and training both the baseline and main model._

In [ ]:
def train(model, n_epochs, train_loader, val_loader, optimizer, device, model_name="model"):

    # Extract parameters for model run name 
    current_lr = optimizer.param_groups[0]['lr']
    current_bs = train_loader.batch_size

    run_name = f"{model_name}_lr{current_lr}_bs{current_bs}"

    # Path setup
    model_dir = os.path.join(PROJECT_ROOT, 'results', 'models')
    os.makedirs(model_dir, exist_ok=True)
    save_path = os.path.join(model_dir, f"best_{run_name}.pth")

    model.to(device)
    criterion = nn.BCEWithLogitsLoss()  

    # LR warm-up
    warmup_epochs = 5
    def warmup_lambda(epoch):
        if epoch < warmup_epochs:
            return float(epoch+1) / float(warmup_epochs)
        return 1.0

    # Scheduler for LR warm-up and cosine annealing LR
    base_scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs)
    warmup_scheduler = LambdaLR(optimizer, lr_lambda=warmup_lambda)

    # Dictionary of metrics for plotting
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [], 'val_kappa': []
    }

    # Early stopping setup
    best_kappa = -1.0
    patience = 50
    patience_counter =0

    for epoch in range(n_epochs):
        # Training phase
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        for batch_idx, (batch_X, batch_y) in enumerate(train_loader):
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            # Ensure float tensor with same shape as preds
            batch_y = batch_y.float().view(-1)

            # Forward pass
            optimizer.zero_grad()
            pred = model(batch_X)
            loss = criterion(pred, batch_y)

            # Backward pass
            loss.backward()

            # Clip gradients to avoid exploiding gradients for Transformer
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()
            predicted_classes = (torch.sigmoid(pred) > 0.5).int()
            train_correct += (predicted_classes == batch_y).sum().item()
            train_total += batch_y.size(0)

        if epoch < warmup_epochs:
            warmup_scheduler.step()
        else:
            base_scheduler.step()

        # Training metrics
        avg_train_loss = train_loss / len(train_loader)
        train_acc = 100 * train_correct / train_total

        metrics = evaluate(model, val_loader, criterion, device)
        history['train_loss'].append(avg_train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(metrics['loss'])
        history['val_acc'].append(metrics['acc'])
        history['val_kappa'].append(metrics['kappa'])

        print(  f"Epoch {epoch+1}/{n_epochs} | Train Loss {avg_train_loss:.4f} | Val Kappa: {metrics['kappa']:.4f}")

        # Early stopping
        if metrics['kappa'] > best_kappa:
            best_kappa = metrics['kappa']
            patience_counter = 0

            torch.save(model.state_dict(), save_path)
            print(f"Saved best model to: {save_path}")

        else:
            patience_counter += 1

        if patience_counter >= patience:
            print(f"Early stopping triggered at epoch {epoch+1}. Best Kappa: {best_kappa:.4f}")
            break

    model.load_state_dict(torch.load(save_path))
    return model, history

In [ ]:
def plot_metrics(history, model_name="model", learning_rate=None, save=False):

    epochs = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(12, 4))

    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], label='Train Loss')
    plt.plot(epochs, history['val_loss'], label='Val Loss')
    plt.title('Loss Curve')
    plt.xlabel('Epochs')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['val_kappa'], label='Val Kappa', color='magenta')
    plt.title('Validation Kappa')
    plt.xlabel('Epochs')
    plt.legend()


    plt.tight_layout()

    if save:
        plot_dir = os.path.join(PROJECT_ROOT, 'results', 'plots')
        os.makedirs(plot_dir, exist_ok=True)

        if learning_rate is not None:
            lr_str = f"lr_{learning_rate:.1e}"
            filename = f"{model_name}_dropout.4_{lr_str}_loss_curve.png"
        else:
            filename = f"{model_name}_dropout.4_loss_curve.png"
        
        save_path = os.path.join(plot_dir, filename)
        plt.savefig(save_path)
        print(f"Plot saved to: {save_path}")
        plt.close()
    else:
        plt.show()

In [ ]:
# Hyperparameter test grid
lrs = [0.001, 0.005, 0.0001]
batch_sizes = [16, 32]

for lr in lrs:
    for bs in batch_sizes:
        print(f"Starting run: LR={lr}, BS={bs}")

        # Create train and test dataloaders
        train_loader, val_loader, test_loader = create_dataloaders(batch_size=bs)

        # Initialize models and optimizers
        model_main = EEGClassifier()
        model_baseline = BaselineCNN()

        optimizer_main = optim.Adam(model_main.parameters(), lr=lr, weight_decay=1e-4)
        optimizer_baseline = optim.Adam(model_baseline.parameters(), lr=lr, weight_decay=1e-4)

        # Training baseline and main model
        trained_main, main_history = train(
            model=model_main, 
            n_epochs=50, 
            train_loader=train_loader, 
            val_loader=val_loader, 
            optimizer=optimizer_main, 
            device=device,
            model_name="main")

        trained_baseline, baseline_history = train(
            model=model_baseline, 
            n_epochs=50, 
            train_loader=train_loader, 
            val_loader=val_loader, 
            optimizer=optimizer_baseline, 
            device=device,
            model_name="baseline_cnn")

        # Evaluating both models
        plot_metrics(baseline_history, model_name="baseline_cnn", learning_rate=lr, save=True)
        plot_metrics(main_history, model_name="main", learning_rate=lr, save=True)

        y_true_baseline, y_pred_baseline = get_predictions(trained_baseline, test_loader, device)
        visualize_predictions(y_true_baseline, y_pred_baseline, model_name="baseline_cnn", learning_rate=lr, save=True)

        y_true_main, y_pred_main = get_predictions(trained_main, test_loader, device)
        visualize_predictions(y_true_main, y_pred_main, model_name="main", learning_rate=lr, save=True)
